# Peak-Motion 3-GOP Feature Extraction Pipeline

**Goal:** For every video in `dataset/Violence` and `dataset/NonViolence`, locate the
GOP (Group of Pictures) chunk with the highest inter-frame motion, grab that chunk plus
its immediate left/right neighbors (3 GOPs = 12 sampled frames), run those 12 frames
through a frozen EfficientNet-B4 backbone, and save the resulting `(12, 1792)` feature
tensor as a single `.pth` file. This is an offline, one-time "1-Go" pass meant to be run
once on Colab (T4 GPU) so that downstream model training reads small cached tensors
instead of decoding video every epoch.

Output mirrors the input class structure:
```
extracted_features_b4/
├── Violence/<video_id>.pth        # torch.Size([12, 1792])
└── NonViolence/<video_id>.pth     # torch.Size([12, 1792])
```

## 0. Install dependencies
Uncomment and run once per fresh Colab runtime. `opencv-python-headless` avoids GUI deps,
`timm` gives us the pretrained EfficientNet-B4 backbone, `psutil` lets us print live RAM
usage for sanity-checking memory stability, `scikit-learn` is included as requested for
any downstream analysis (e.g. train/test splitting on top of the cached tensors).

In [ ]:
!pip install -q timm psutil opencv-python-headless scikit-learn

## 0.5 Mount Google Drive
Required so that `/content/drive/MyDrive/...` paths in `CONFIG` resolve. Colab will
prompt for an auth code the first time this runs in a fresh runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Imports & global configuration
All tunable knobs live in one `CONFIG` dict so the rest of the notebook never hardcodes
magic numbers. This also makes it trivial to re-run on a different dataset path or a
different GOP size without touching the pipeline logic.

In [ ]:
import os
import gc
import json
import time
import traceback
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn as nn
import timm
import psutil

# ----------------------------------------------------------------------------------------
# CONFIG: single source of truth for every tunable parameter in the pipeline.
# ----------------------------------------------------------------------------------------
CONFIG = {
    # Input dataset root (Google Drive mount point). Expected layout:
    #   research/dataset/Violence/*.mp4|*.avi
    #   research/dataset/NonViolence/*.mp4|*.avi
    "input_root": "/content/drive/MyDrive/research/dataset",
    "class_names": ["Violence", "NonViolence"],

    # Output root (Google Drive mount point). Mirrors input_root's class subfolders,
    # but holds .pth feature files.
    "output_root": "/content/drive/MyDrive/research/features",

    # Video extensions we will attempt to process.
    "video_extensions": (".mp4", ".avi", ".mov", ".mkv"),

    # GOP (Group of Pictures) chunk size. Most consumer / surveillance footage uses a GOP
    # of ~15 frames for a 30fps stream (i.e. one keyframe every 0.5s). We *also* support
    # dynamically recomputing this if a video's frame count is small relative to this size.
    "gop_size": 15,

    # Number of frames uniformly sampled from *each* GOP chunk for motion scoring AND for
    # final feature extraction. Per the spec: 4 frames per GOP x 3 GOPs (prev/peak/next) = 12.
    "frames_per_gop": 4,

    # Number of GOPs in the final neighborhood window (must stay 3 -> 12 total frames).
    "neighborhood_gops": 3,

    # Spatial resolution fed into EfficientNet-B4 (its native training resolution).
    "resize_dim": (380, 380),

    # Backbone. `features_only=True` + `out_indices=(-1,)` gives us the last conv feature
    # map (pre-pooling) with 1792 channels for the B4 variant.
    "backbone_name": "efficientnet_b4",

    # Where to log videos we could not process (corrupt, 0 frames, decode errors, etc).
    # Pointed at Drive (not local Colab storage) so the log survives runtime resets.
    "error_log_path": "/content/drive/MyDrive/research/features/extraction_errors.json",

    # Device selection happens at runtime (T4 GPU in Colab -> "cuda").
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    # Batch size for the single forward pass per video (always 12 frames -> 1 batch).
    "feature_dim": 448,
    "target_seq_len": 12,
}

print("Configuration loaded:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")
print(f"\nDevice selected: {CONFIG['device']}")
if CONFIG["device"] == "cpu":
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > T4 GPU in Colab.")

## 2. Backbone model setup
We load a frozen (`eval()` + `requires_grad_(False)`) EfficientNet-B4 in `features_only`
mode so `timm` returns raw convolutional feature maps instead of pooled/classified
outputs. We then apply our own adaptive average pool down to a single spatial vector per
frame, collapsing each frame's feature map to a `(1792,)` vector. Stacking 12 frames
yields the required `(12, 1792)` tensor.

In [ ]:
def build_backbone(config: dict) -> nn.Module:
    """
    Build a frozen EfficientNet-B4 feature extractor.

    Returns the last pre-pooling convolutional feature map per frame. We additionally
    wrap it with global average pooling so each frame collapses to a single 1792-d
    vector, matching the required (12, 1792) output shape per video.
    """
    model = timm.create_model(
        config["backbone_name"],
        pretrained=True,
        features_only=True,
        out_indices=(-1,),  # last feature stage -> 1792 channels for B4
    )
    model.eval()
    for param in model.parameters():
        param.requires_grad_(False)
    model.to(config["device"])

    feat_channels = model.feature_info.channels()[-1]
    if feat_channels != config["feature_dim"]:
        print(
            f"WARNING: backbone reports {feat_channels} channels, "
            f"expected {config['feature_dim']}. Continuing, but downstream shapes "
            f"will reflect the actual value."
        )
    return model


print("Loading frozen EfficientNet-B4 backbone (pretrained on ImageNet)...")
backbone = build_backbone(CONFIG)
gap = nn.AdaptiveAvgPool2d(1)  # spatial collapse: (B, C, H, W) -> (B, C, 1, 1)
print("Backbone ready and frozen.")

## 3. Core pipeline functions
Each function below does exactly one job, so the per-video loop in Section 5 reads like
plain English. All functions are defensive: they validate shapes/lengths and raise
clear, catchable exceptions rather than silently producing malformed tensors.

### 3.1 Frame decoding + GOP chunking
We decode the *entire* video into a single in-memory list of frames (acceptable for
short clips typical of violence-detection datasets, generally a few seconds long), then
slice that list into fixed-size GOP chunks. The last chunk may be shorter than
`gop_size` if the frame count doesn't divide evenly — that's fine, later steps tolerate
uneven chunk lengths.

In [ ]:
def decode_video_to_gop_chunks(video_path: str, gop_size: int):
    """
    Decode a video with OpenCV and split its frames into GOP-sized chunks.

    Returns:
        chunks: List[List[np.ndarray]] - list of GOP chunks, each a list of BGR frames.
        total_frames: int

    Raises:
        RuntimeError if the file can't be opened or yields zero frames.
    """
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        cap.release()
        raise RuntimeError(f"OpenCV could not open video: {video_path}")

    frames = []
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frames.append(frame)
    finally:
        cap.release()  # always release the capture handle, even on error

    if len(frames) == 0:
        raise RuntimeError(f"Video decoded but contains 0 readable frames: {video_path}")

    # Dynamic GOP size: if the whole video is shorter than one nominal GOP, shrink the
    # GOP size so we still get at least a couple of chunks to work with, down to a floor
    # of 1 frame/chunk for extremely short clips.
    effective_gop_size = gop_size
    if len(frames) < gop_size * 2:
        effective_gop_size = max(1, len(frames) // 3) or 1

    chunks = [
        frames[i: i + effective_gop_size]
        for i in range(0, len(frames), effective_gop_size)
    ]
    # Drop any trailing empty chunk artifact (shouldn't happen, but stay defensive).
    chunks = [c for c in chunks if len(c) > 0]

    total_frames = len(frames)
    del frames  # frames now live only inside `chunks`; drop the flat reference promptly
    return chunks, total_frames

### 3.2 Uniform sampling + motion scoring
For each GOP we uniformly pick `frames_per_gop` indices (np.linspace), then compute the
motion score as the sum of absolute pixel differences between consecutive sampled
frames: `score = sum(|F1-F0| + |F2-F1| + |F3-F2|)`. If a chunk has fewer than
`frames_per_gop` raw frames, we sample with repetition so scoring never crashes.

In [ ]:
def uniform_sample_indices(n_available: int, n_target: int):
    """
    Uniformly pick `n_target` indices from a sequence of length `n_available`.
    If n_available < n_target, indices repeat (sampling with repetition) so the
    caller always gets exactly n_target indices, even from a 1-frame chunk.
    """
    if n_available <= 0:
        raise RuntimeError("Cannot sample indices from an empty chunk.")
    if n_available >= n_target:
        return np.linspace(0, n_available - 1, n_target).round().astype(int)
    # Fewer frames than needed: repeat indices cyclically to reach n_target.
    base = np.arange(n_available)
    reps = int(np.ceil(n_target / n_available))
    return np.tile(base, reps)[:n_target]


def sample_gop_frames(chunk, frames_per_gop: int):
    """Return exactly `frames_per_gop` BGR frames uniformly sampled from a GOP chunk."""
    idxs = uniform_sample_indices(len(chunk), frames_per_gop)
    return [chunk[i] for i in idxs]


def compute_motion_score(sampled_frames):
    """
    Score = sum(|F1-F0| + |F2-F1| + |F3-F2| + ...) computed on LUMINANCE only.
    Converting to grayscale first removes chromatic/compression noise from the score
    and is ~3x cheaper than diffing all 3 BGR channels. The previous frame's grayscale
    image is reused each iteration instead of being reconverted from scratch.
    """
    if len(sampled_frames) < 2:
        return 0.0  # a single-frame "chunk" has no internal motion by definition

    score = 0.0
    prev_gray = cv2.cvtColor(sampled_frames[0], cv2.COLOR_BGR2GRAY)
    for curr_f in sampled_frames[1:]:
        curr_gray = cv2.cvtColor(curr_f, cv2.COLOR_BGR2GRAY)
        diff = cv2.absdiff(curr_gray, prev_gray)  # single-channel, uint8-safe
        score += float(np.sum(diff, dtype=np.float64))
        prev_gray = curr_gray  # reuse instead of reconverting next iteration
    return score

### 3.3 Peak-GOP localization + 3-GOP neighborhood extraction
We score every GOP, find the index of the maximum, then grab `[peak-1, peak, peak+1]`.
Boundary handling: if the peak is the first or last chunk, the window is shifted inward
rather than indexing out of bounds. If there aren't even 3 chunks total, we fall back to
frame-cloning (Section 3.4) to stretch whatever we have up to 12 frames.

In [ ]:
def select_neighborhood_chunk_indices(num_chunks: int, peak_idx: int, window: int = 3):
    """
    Given the total number of GOP chunks and the peak-motion chunk index, return the
    list of chunk indices forming a `window`-sized neighborhood centered on the peak,
    shifted inward at boundaries so it never runs out of bounds.

    Returns None if num_chunks < window (caller should use frame-cloning fallback instead).
    """
    if num_chunks < window:
        return None

    half = window // 2
    start = peak_idx - half
    end = start + window  # exclusive

    # Shift the window inward if it spills past either boundary.
    if start < 0:
        shift = -start
        start += shift
        end += shift
    if end > num_chunks:
        shift = end - num_chunks
        start -= shift
        end -= shift

    # Final safety clamp (covers num_chunks == window exactly).
    start = max(0, start)
    end = min(num_chunks, end)
    return list(range(start, end))

### 3.4 Uniform frame-cloning fallback
Used whenever we cannot naturally gather 12 distinct sampled frames (videos with < 3
GOPs, or extremely short clips). We take whatever frames *are* available, uniformly
sampled in their natural order, and cyclically repeat them until we have exactly
`target_len` frames. This preserves temporal order while guaranteeing the strict output
shape contract `(12, 1792)`.

In [ ]:
def clone_frames_to_length(frames, target_len: int):
    """Cyclically repeat `frames` until the list has exactly `target_len` entries."""
    if len(frames) == 0:
        raise RuntimeError("Cannot clone an empty frame list.")
    if len(frames) >= target_len:
        return frames[:target_len]
    reps = int(np.ceil(target_len / len(frames)))
    stretched = (frames * reps)[:target_len]
    return stretched

### 3.5 Backbone inference
Takes exactly 12 BGR frames, resizes + normalizes them into a single batch, runs the
frozen backbone once, and global-average-pools each frame's feature map down to a
`(1792,)` vector, returning the stacked `(12, 1792)` tensor on CPU (ready for
`torch.save`).

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def frames_to_feature_tensor(frames, backbone_model, gap_layer, config):
    """
    frames: list of exactly 12 BGR np.ndarray frames (H, W, 3), uint8.
    Returns: torch.Tensor of shape (12, feature_dim) on CPU.
    """
    assert len(frames) == config["target_seq_len"], (
        f"Expected exactly {config['target_seq_len']} frames, got {len(frames)}"
    )

    resized = []
    for f in frames:
        f_rgb = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        f_resized = cv2.resize(f_rgb, config["resize_dim"], interpolation=cv2.INTER_AREA)
        resized.append(f_resized)

    # Stack as a compact uint8 array on CPU (cheap, no float math here).
    batch = np.stack(resized, axis=0)  # (12, H, W, 3) uint8

    # Move the small uint8 tensor to GPU first, THEN cast to float32 on-device.
    # This avoids the CPU ever materializing a large float32 array.
    batch_t = torch.from_numpy(batch).permute(0, 3, 1, 2).to(
        config["device"], dtype=torch.float32
    )

    # Normalization (/255 + mean/std subtraction) now runs entirely on GPU in parallel.
    mean = IMAGENET_MEAN.to(config["device"])
    std = IMAGENET_STD.to(config["device"])
    batch_t = (batch_t / 255.0 - mean) / std

    with torch.no_grad():
        feat_maps = backbone_model(batch_t)[-1]   # (12, 1792, h, w) - last stage only
        pooled = gap_layer(feat_maps)              # (12, 1792, 1, 1)
        pooled = pooled.flatten(1)                  # (12, 1792)

    result = pooled.detach().to("cpu")

    # Explicitly drop GPU-side intermediates before returning, the caller will also
    # call torch.cuda.empty_cache() once per video to keep VRAM flat across the loop.
    del batch_t, feat_maps, pooled
    return result

## 4. Single-video orchestration
Glues all of Section 3 together for one video file: decode -> chunk -> score -> locate
peak -> gather neighborhood (or clone-fallback) -> sample 4 frames/GOP -> backbone ->
save `.pth`. Every failure mode raises a descriptive exception that the outer loop in
Section 5 catches and logs, so one bad file never halts the whole run.

In [ ]:
def process_single_video(video_path: str, out_path: str, backbone_model, gap_layer, config):
    """
    Run the full peak-motion 3-GOP extraction pipeline on one video.

    Returns a small dict of metadata (peak_gop_idx, num_gops, used_cloning) for logging.
    Raises on any unrecoverable failure (caller is responsible for catching + logging).
    """
    chunks, total_frames = decode_video_to_gop_chunks(video_path, config["gop_size"])
    num_chunks = len(chunks)

    # --- Step 1: uniformly sample 4 frames from every GOP + score motion per GOP -------
    per_gop_samples = []   # List[List[frame]] - 4 sampled frames per GOP, in chunk order
    motion_scores = []
    for chunk in chunks:
        sampled = sample_gop_frames(chunk, config["frames_per_gop"])
        per_gop_samples.append(sampled)
        motion_scores.append(compute_motion_score(sampled))

    peak_gop_idx = int(np.argmax(motion_scores))
    used_cloning = False

    # --- Step 2: gather the 3-GOP neighborhood around the peak -------------------------
    neighborhood_idxs = select_neighborhood_chunk_indices(
        num_chunks, peak_gop_idx, window=config["neighborhood_gops"]
    )

    if neighborhood_idxs is not None:
        # Happy path: enough GOPs exist, flatten their pre-sampled frames in order.
        final_frames = []
        for idx in neighborhood_idxs:
            final_frames.extend(per_gop_samples[idx])
    else:
        # Fallback path: fewer than 3 GOPs total. Flatten everything we have, in order,
        # then cyclically clone up to the exact target length.
        used_cloning = True
        final_frames = []
        for sampled in per_gop_samples:
            final_frames.extend(sampled)

    # Whether from the happy path or the fallback, the length must land on exactly 12.
    # The happy path with 3 full GOPs * 4 frames already yields 12; any shortfall
    # (e.g. a GOP that itself had < 4 raw frames so sampling repeated) is also handled
    # safely here, since clone_frames_to_length is a no-op once length == target.
    if len(final_frames) != config["target_seq_len"]:
        used_cloning = True
        final_frames = clone_frames_to_length(final_frames, config["target_seq_len"])

    # --- Step 3: backbone inference -> (12, 1792) tensor --------------------------------
    feature_tensor = frames_to_feature_tensor(final_frames, backbone_model, gap_layer, config)

    # Strict, absolute shape check — no soft `or` fallback. Any truncated or unexpected
    # dimension fails loudly here instead of silently reaching disk.
    assert feature_tensor.shape == (config["target_seq_len"], config["feature_dim"]), (
        f"Feature tensor shape mismatch for '{video_path}': "
        f"expected {(config['target_seq_len'], config['feature_dim'])}, "
        f"got {tuple(feature_tensor.shape)}"
    )

    # --- Step 4: persist to disk ---------------------------------------------------------
    torch.save(feature_tensor, out_path)

    meta = {
        "num_gops": num_chunks,
        "total_frames": total_frames,
        "peak_gop_idx": peak_gop_idx,
        "used_cloning_fallback": used_cloning,
        "output_shape": list(feature_tensor.shape),
    }

    # --- Step 5: aggressive cleanup so RAM/VRAM stay flat across thousands of videos ----
    del chunks, per_gop_samples, motion_scores, final_frames, feature_tensor
    gc.collect()
    if config["device"] == "cuda":
        torch.cuda.empty_cache()

    return meta

## 5. Dataset-wide driver loop
Walks `input_root/<class_name>/*` for every configured video extension, builds the
mirrored output path, skips files that are already processed (resumable!), and calls
`process_single_video` inside a try/except so corrupt files are logged and skipped
rather than crashing the whole Colab session. Prints a clean one-line progress status
per video, including live RAM usage so memory stability is visible in real time.

In [ ]:
def discover_videos(config):
    """Walk the input dataset and return a flat list of (class_name, video_path) tuples."""
    video_list = []
    input_root = Path(config["input_root"])
    for class_name in config["class_names"]:
        class_dir = input_root / class_name
        if not class_dir.is_dir():
            print(f"WARNING: expected class directory not found, skipping: {class_dir}")
            continue
        for ext in config["video_extensions"]:
            video_list.extend(sorted(class_dir.glob(f"*{ext}")))
            video_list.extend(sorted(class_dir.glob(f"*{ext.upper()}")))
    # Re-pair with class names (derived from parent folder) and de-duplicate.
    seen = set()
    paired = []
    for p in video_list:
        if p in seen:
            continue
        seen.add(p)
        paired.append((p.parent.name, p))
    return paired


def run_extraction(config, backbone_model, gap_layer):
    out_root = Path(config["output_root"])
    for class_name in config["class_names"]:
        (out_root / class_name).mkdir(parents=True, exist_ok=True)

    videos = discover_videos(config)
    total = len(videos)
    print(f"Discovered {total} candidate video files across {config['class_names']}.\n")

    errors = []
    success_count = 0
    skip_count = 0
    start_time = time.time()

    for i, (class_name, video_path) in enumerate(videos, start=1):
        video_id = video_path.stem
        out_path = out_root / class_name / f"{video_id}.pth"

        # Resumability: skip videos already successfully processed in a prior run.
        if out_path.exists():
            skip_count += 1
            print(f"[{i}/{total}] {video_path.name:<45} | Status: SKIP (already exists)")
            continue

        try:
            meta = process_single_video(
                str(video_path), str(out_path), backbone_model, gap_layer, config
            )
            success_count += 1
            ram_pct = psutil.virtual_memory().percent
            print(
                f"[{i}/{total}] Video: {video_path.name:<40} | "
                f"Peak GOP: {meta['peak_gop_idx']:>3} / {meta['num_gops']:<3} | "
                f"Cloned: {str(meta['used_cloning_fallback']):<5} | "
                f"RAM: {ram_pct:>4.1f}% | Status: OK"
            )
        except Exception as exc:
            error_entry = {
                "video_path": str(video_path),
                "class_name": class_name,
                "error": str(exc),
                "traceback": traceback.format_exc(),
            }
            errors.append(error_entry)
            print(
                f"[{i}/{total}] Video: {video_path.name:<40} | "
                f"Status: FAILED -> {exc}"
            )
            # Best-effort cleanup even on failure, so a corrupt file doesn't leak memory.
            gc.collect()
            if config["device"] == "cuda":
                torch.cuda.empty_cache()
            continue

    elapsed = time.time() - start_time

    # Persist the error log (even if empty, for a consistent audit trail).
    with open(config["error_log_path"], "w") as f:
        json.dump(errors, f, indent=2)

    return {
        "total_discovered": total,
        "succeeded": success_count,
        "skipped_existing": skip_count,
        "failed": len(errors),
        "elapsed_seconds": elapsed,
    }

## 6. Run the extraction
This is the single cell that actually executes the full pass over the dataset. Re-running
it after an interruption is safe: already-written `.pth` files are detected and skipped.

In [ ]:
summary = run_extraction(CONFIG, backbone, gap)

## 7. Final verification report
Counts the actual `.pth` files written per class on disk (ground truth, not just the
in-memory counters) and cross-checks against the run summary, so any silent filesystem
issue is caught immediately.

In [ ]:
def print_final_report(config, summary):
    out_root = Path(config["output_root"])
    print("=" * 70)
    print("EXTRACTION COMPLETE - FINAL REPORT")
    print("=" * 70)
    print(f"Total videos discovered : {summary['total_discovered']}")
    print(f"Succeeded this run      : {summary['succeeded']}")
    print(f"Skipped (already done)  : {summary['skipped_existing']}")
    print(f"Failed / logged errors  : {summary['failed']}")
    print(f"Elapsed time            : {summary['elapsed_seconds']:.1f}s")
    print("-" * 70)

    grand_total = 0
    for class_name in config["class_names"]:
        class_dir = out_root / class_name
        n_files = len(list(class_dir.glob("*.pth"))) if class_dir.is_dir() else 0
        grand_total += n_files
        print(f"  {class_name:<15} -> {n_files} .pth files written to {class_dir}")
    print("-" * 70)
    print(f"  TOTAL .pth files on disk: {grand_total}")

    if summary["failed"] > 0:
        print(
            f"\n{summary['failed']} video(s) failed and were logged to "
            f"'{config['error_log_path']}'. Inspect that file for details."
        )
    print("=" * 70)


print_final_report(CONFIG, summary)